# SPFC Evaluation Bench - Kaggle

Kaggle runbook for SPFC vs Rectified-CFG++ vs base SD3 Medium. Every generation/evaluation run is isolated in its own cell and displays a rough time estimate plus elapsed time.

In [ ]:
import os

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("DIFFUSERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

GITHUB_REPO_URL = 'https://github.com/Soobiwan/aim-flow.git'
SEED = 13
RUN_ROOT = '/kaggle/working/spfc_eval_seed13/runs'
REPORT_DIR = '/kaggle/working/spfc_eval_seed13/reports'
EVAL_DIR = f'{REPORT_DIR}/eval'
T2I_MANIFEST = '/kaggle/working/aim-flow/benchmarks/manifests/t2i_compbench_100_seed13.json'
COCO_MANIFEST = '/kaggle/working/aim-flow/benchmarks/manifests/coco_100_seed13.json'
T2I_DECOMP = '/kaggle/working/aim-flow/benchmarks/decompositions/t2i_compbench_100_seed13_spfc.json'
COCO_DECOMP = '/kaggle/working/aim-flow/benchmarks/decompositions/coco_100_seed13_spfc.json'
T2I_DATASET_ROOT = '/kaggle/working/aim-flow/external/T2I-CompBench/examples/dataset'
COCO_PROMPT_FILE = ''
EXECUTE_T2I_OFFICIAL = False
QUALITATIVE_MANIFEST = T2I_MANIFEST

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/aim-flow
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

In [ ]:
# If Kaggle gives a P100 with an incompatible Torch build, uncomment these and restart the runtime.
# !pip uninstall -y torch torchvision torchaudio
# !pip install --no-cache-dir --force-reinstall torch==2.4.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -q -r requirements-kaggle.txt
!pip install -q --no-deps -e .


In [ ]:
import os
from IPython.display import Markdown, display

MODEL_ID = 'stabilityai/stable-diffusion-3-medium-diffusers'
HF_SECRET_NAMES = ('Huggingface', 'HF_TOKEN', 'HUGGINGFACE_TOKEN')

if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in HF_SECRET_NAMES:
            try:
                token = secrets.get_secret(secret_name)
            except Exception:
                token = None
            if token:
                os.environ['HF_TOKEN'] = token
                break
    except Exception:
        pass

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not token:
    raise RuntimeError(
        'Missing Hugging Face token. In Kaggle, add a secret named Huggingface or HF_TOKEN, '
        'turn it on for this notebook, and make sure that Hugging Face account has accepted the SD3 Medium license.'
    )

try:
    from huggingface_hub import HfApi
    HfApi().model_info(MODEL_ID, token=token)
except Exception as exc:
    raise RuntimeError(
        f'HF_TOKEN is set, but access check for {MODEL_ID} failed. '
        'Confirm the Kaggle secret is enabled and the token account has accepted the gated model license.'
    ) from exc

display(Markdown('Hugging Face token is configured and can access SD3 Medium.'))


In [ ]:
import json, os, subprocess, time
from pathlib import Path
from IPython.display import Image as DisplayImage, Markdown, display

EST_SEC_PER_PROMPT = {
    'spfc_generation': 240,
    'rectified_cfgpp_generation': 90,
    'base_generation': 45,
    't2i_official_eval': 8,
    't2i_stage_only': 0.05,
    'coco_clip_eval': 1.2,
}

def manifest_count(path, default=100):
    path = Path(path)
    if not path.exists():
        return default
    return len(json.loads(path.read_text(encoding='utf-8'))['samples'])

def fmt_seconds(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}h {m}m {s}s' if h else f'{m}m {s}s'

def timed_run(label, command, estimated_seconds):
    display(Markdown(f'### {label}\nEstimated time: **{fmt_seconds(estimated_seconds)}**'))
    start = time.perf_counter()
    env = os.environ.copy()
    env.setdefault('PYTHONUNBUFFERED', '1')
    process = subprocess.Popen(
        command,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    returncode = process.wait()
    elapsed = time.perf_counter() - start
    display(Markdown(f'Finished **{label}** in **{fmt_seconds(elapsed)}**.'))
    if returncode != 0:
        raise RuntimeError(f'Command failed with exit code {returncode}: {command}')

def show_scores(path):
    path = Path(path)
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        display(Markdown('```json\n' + json.dumps(data.get('scores', data), indent=2) + '\n```'))

def show_markdown(path):
    path = Path(path)
    if path.exists():
        display(Markdown(path.read_text(encoding='utf-8')))

def show_image(path):
    path = Path(path)
    if path.exists():
        display(DisplayImage(filename=str(path)))

In [ ]:
timed_run('Prepare T2I-CompBench 100-prompt manifest and SPFC template', f'python scripts/bench_prepare_prompts.py --benchmark t2i_compbench --seed {SEED} --t2i-subset-size 100 --t2i-dataset-root {T2I_DATASET_ROOT} --write-decomposition-template', 5)

In [ ]:
coco_args = f'--benchmark coco --seed {SEED} --coco-subset-size 100 --write-decomposition-template'
if COCO_PROMPT_FILE:
    coco_args += f' --coco-prompt-file {COCO_PROMPT_FILE}'
timed_run('Prepare COCO 100-prompt manifest and SPFC template', f'python scripts/bench_prepare_prompts.py {coco_args}', 180)

Replace the generated decomposition templates with real LLM/manual SPFC decompositions before generation.

In [ ]:
timed_run('Validate T2I SPFC decompositions', f'python scripts/bench_validate_decompositions.py --manifest {T2I_MANIFEST} --decompositions {T2I_DECOMP}', 2)

In [ ]:
timed_run('Validate COCO SPFC decompositions', f'python scripts/bench_validate_decompositions.py --manifest {COCO_MANIFEST} --decompositions {COCO_DECOMP}', 2)

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('SPFC T2I-CompBench generation', f'python scripts/bench_generate.py --manifest {T2I_MANIFEST} --decompositions {T2I_DECOMP} --run-root {RUN_ROOT} --methods spfc --seed {SEED} --skip-existing', N * EST_SEC_PER_PROMPT['spfc_generation'])

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('Rectified-CFG++ T2I-CompBench generation', f'python scripts/bench_generate.py --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --methods rectified_cfgpp --seed {SEED} --skip-existing', N * EST_SEC_PER_PROMPT['rectified_cfgpp_generation'])

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('Base SD3 T2I-CompBench generation', f'python scripts/bench_generate.py --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --methods base --seed {SEED} --guidance-scale 1.0 --skip-existing', N * EST_SEC_PER_PROMPT['base_generation'])

In [ ]:
N = manifest_count(COCO_MANIFEST)
timed_run('SPFC COCO generation', f'python scripts/bench_generate.py --manifest {COCO_MANIFEST} --decompositions {COCO_DECOMP} --run-root {RUN_ROOT} --methods spfc --seed {SEED} --skip-existing', N * EST_SEC_PER_PROMPT['spfc_generation'])

In [ ]:
N = manifest_count(COCO_MANIFEST)
timed_run('Rectified-CFG++ COCO generation', f'python scripts/bench_generate.py --manifest {COCO_MANIFEST} --run-root {RUN_ROOT} --methods rectified_cfgpp --seed {SEED} --skip-existing', N * EST_SEC_PER_PROMPT['rectified_cfgpp_generation'])

In [ ]:
N = manifest_count(COCO_MANIFEST)
timed_run('Base SD3 COCO generation', f'python scripts/bench_generate.py --manifest {COCO_MANIFEST} --run-root {RUN_ROOT} --methods base --seed {SEED} --guidance-scale 1.0 --skip-existing', N * EST_SEC_PER_PROMPT['base_generation'])

In [ ]:
for path in [Path(EVAL_DIR) / 't2i_compbench_scores.json', Path(EVAL_DIR) / 'coco_scores.json']:
    if path.exists():
        path.unlink()

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('SPFC T2I-CompBench evaluation', f'python scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods spfc --append {flag}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('Rectified-CFG++ T2I-CompBench evaluation', f'python scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods rectified_cfgpp --append {flag}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('Base SD3 T2I-CompBench evaluation', f'python scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods base --append {flag}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
N = manifest_count(COCO_MANIFEST)
timed_run('SPFC COCO CLIPScore evaluation', f'python scripts/bench_evaluate.py --benchmark coco --manifest {COCO_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods spfc --append', N * EST_SEC_PER_PROMPT['coco_clip_eval'] + 90)
show_scores(Path(EVAL_DIR) / 'coco_scores.json')

In [ ]:
N = manifest_count(COCO_MANIFEST)
timed_run('Rectified-CFG++ COCO CLIPScore evaluation', f'python scripts/bench_evaluate.py --benchmark coco --manifest {COCO_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods rectified_cfgpp --append', N * EST_SEC_PER_PROMPT['coco_clip_eval'] + 90)
show_scores(Path(EVAL_DIR) / 'coco_scores.json')

In [ ]:
N = manifest_count(COCO_MANIFEST)
timed_run('Base SD3 COCO CLIPScore evaluation', f'python scripts/bench_evaluate.py --benchmark coco --manifest {COCO_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods base --append', N * EST_SEC_PER_PROMPT['coco_clip_eval'] + 90)
show_scores(Path(EVAL_DIR) / 'coco_scores.json')

In [ ]:
T2I_SCORES = f'{EVAL_DIR}/t2i_compbench_scores.json'
COCO_SCORES = f'{EVAL_DIR}/coco_scores.json'
QUAL_GRID = f'{REPORT_DIR}/qualitative_grid.png'
timed_run('Build tables and qualitative grid', f'python scripts/bench_report.py --t2i-scores {T2I_SCORES} --coco-scores {COCO_SCORES} --coco-manifest {COCO_MANIFEST} --run-root {RUN_ROOT} --output-dir {REPORT_DIR} --qualitative-manifest {QUALITATIVE_MANIFEST} --qualitative-output {QUAL_GRID}', 10)
show_markdown(Path(REPORT_DIR) / 't2i_compbench_table.md')
show_markdown(Path(REPORT_DIR) / 'coco_table.md')
show_image(QUAL_GRID)

In [ ]:
PROBE_DIR = '/kaggle/working/spfc_eval_seed13/probes/fox_rainboots_seed13'
FOX_PROMPT_KEY = 'marble_fox_rainboots_mirror'
FOX_PROBE_MAX_STEPS = None  # set to 4 for a quick smoke run before the full 16-step probe
FOX_PROBE_LIMIT_ARG = f'--max-probe-steps {FOX_PROBE_MAX_STEPS}' if FOX_PROBE_MAX_STEPS else ''
timed_run('Fox rainboots SPFC cutoff-final probe', f'python scripts/bench_probe_fox.py --prompt-key {FOX_PROMPT_KEY} --probe-mode cutoff {FOX_PROBE_LIMIT_ARG} --output-dir {PROBE_DIR}', (FOX_PROBE_MAX_STEPS or 16) * EST_SEC_PER_PROMPT['spfc_generation'])
show_image(f'{PROBE_DIR}/fox_cutoff_final_grid.png')

In [ ]:
timed_run('Fox rainboots SPFC step-rollout probe', f'python scripts/bench_probe_fox.py --prompt-key {FOX_PROMPT_KEY} --probe-mode rollout {FOX_PROBE_LIMIT_ARG} --output-dir {PROBE_DIR}', EST_SEC_PER_PROMPT['spfc_generation'] + (FOX_PROBE_MAX_STEPS or 16) * EST_SEC_PER_PROMPT['base_generation'])
show_image(f'{PROBE_DIR}/fox_step_rollout_grid.png')